### Week 5-2 Practice: Broadcasting

**Note:** Fill in every `# YOUR CODE HERE`. Each problem has `assert` cells you can run to check your own work before moving on.

In [1]:
import numpy as np

#### Warm-up: predict the shape, then check
For each `(shape_a, shape_b)` pair below, decide -- on paper, or out loud with a partner -- whether `a + b` would work, and if so, what shape the result would be. *Then* run the next cell to check yourselves.

In [2]:
pairs = [
    ((3, 4), (4,)),
    ((3, 4), (3,)),
    ((3, 4), (3, 1)),
    ((3, 1), (1, 4)),
    ((2, 3, 4), (3, 1)),
    ((5,), (5, 1)),
]
# no, no, yes(2), yes(2), no, no

In [3]:
for a, b in pairs:
    try:
        shape = (np.zeros(a) + np.zeros(b)).shape
        print(f"{a} + {b}  ->  works, result shape {shape}")
    except ValueError:
        print(f"{a} + {b}  ->  ValueError, cannot broadcast")

(3, 4) + (4,)  ->  works, result shape (3, 4)
(3, 4) + (3,)  ->  ValueError, cannot broadcast
(3, 4) + (3, 1)  ->  works, result shape (3, 4)
(3, 1) + (1, 4)  ->  works, result shape (3, 4)
(2, 3, 4) + (3, 1)  ->  works, result shape (2, 3, 4)
(5,) + (5, 1)  ->  works, result shape (5, 5)


#### Problem 1: Curve the grades
A small gradebook: 5 students, 3 exams.

**(a)** Curve every exam by a *different* amount: exam 1 gets +3, exam 2 gets +5, exam 3 gets +0. Do it with one broadcasted expression -- no loops.

In [4]:
scores = np.array([[62, 71, 88],
                    [75, 60, 95],
                    [90, 85, 70],
                    [55, 65, 60],
                    [80, 90, 100]])
exam_bonus = np.array([3, 5, 0])

curved_by_exam = scores + exam_bonus


In [5]:
assert curved_by_exam.shape == (5, 3), "Check the shape"
assert np.array_equal(curved_by_exam[0], [65, 76, 88]), "Check the values"
assert np.array_equal(curved_by_exam[:, 2], scores[:, 2]), "Exam 3 should be unchanged (bonus 0)"
print("Part (a) looks right!")

Part (a) looks right!


**(b)** Now curve each *student* by a different amount instead, so every one of a student's three exam scores gets that student's bonus. First try `scores + student_bonus` -- what happens, and why? Then fix it.

In [6]:
student_bonus = np.array([2, 0, 4, 10, 1])

# First try this in a scratch cell and see what happens:
#     scores + student_bonus

# Now fix it:
# curved_by_student = ...
curved_by_student = (scores.T+student_bonus).T

In [7]:
assert curved_by_student.shape == (5, 3), "Check the shape"
assert np.array_equal(curved_by_student[3], scores[3] + 10), "Student 4 (index 3) should get +10 on every exam"
assert np.array_equal(curved_by_student[1], scores[1]), "Student 2 (index 1) should get +0"
print("Part (b) looks right!")

Part (b) looks right!


#### Problem 2: Combine two 1-D arrays into a grid
4 students each have a score; 3 exams each have a difficulty score. We want a `(4, 3)` grid `predicted` where `predicted[i, j] = ability[i] + difficulty[j]` -- the same `(4, 1) + (1, 3) -> (4, 3)` broadcasting picture from lecture, but now you build it yourself, from two plain 1-D arrays.

In [10]:
ability = np.array([70, 85, 60, 95])   # one score per student, shape (4,)
difficulty = np.array([-5, 0, 10])      # one score per exam, shape (3,)

# First try this in a scratch cell and see what happens (and why):
# ability + difficulty

# Now give each array the shape it needs and build predicted, shape (4, 3):
predicted = ability.reshape((4,1))+difficulty.reshape((1,3))


In [11]:
assert predicted.shape == (4, 3), "Check the shape"
assert np.array_equal(predicted[:, 0], ability - 5), "Column 0 should be ability + difficulty[0]"
assert np.array_equal(predicted[0, :], ability[0] + difficulty), "Row 0 should be ability[0] + difficulty"
print("Problem 2 looks right!")

Problem 2 looks right!


#### Problem 3: Build a design matrix
`design_matrix(x)` should return an `(n, 2)` array whose first column is all ones (the intercept) and whose second column is `x`. `predict(X, beta)` should return the fitted values -- but compute it with elementwise multiplication and `.sum(axis=1)` instead of `@`, to practice broadcasting.

In [12]:
def design_matrix(x):
    """
    Given a 1-D array x of length n, return an (n, 2) array whose first
    column is all ones and whose second column is x.
    """
    a = np.ones((len(x),2))
    a[:,1] = x
    return a

def predict(X, beta):
    """
    Given a design matrix X of shape (n, 2) and coefficients beta of shape
    (2,), return the fitted values of shape (n,). Do not use @ or np.dot;
    use elementwise multiplication and .sum(axis=1) instead.
    """
    return (X*beta).sum(axis=1)


In [14]:
x = np.array([2.0, 4.0, 6.0, 8.0])
beta = np.array([1.0, 0.5])

X = design_matrix(x)
assert X.shape == (4, 2), "design_matrix should return an (n, 2) array"
assert np.array_equal(X[:, 0], np.ones(4)), "First column should be all ones"
assert np.array_equal(X[:, 1], x), "Second column should be x"

yhat = predict(X, beta)
assert np.allclose(yhat, X @ beta), "predict should match X @ beta"
assert np.allclose(yhat, [2.0, 3.0, 4.0, 5.0])
print("All design_matrix/predict tests passed!")

All design_matrix/predict tests passed!


**Optional:** build a 9x9 multiplication table with a single broadcasting expression, no loops.

In [15]:
table = np.arange(1, 10)[:, None] * np.arange(1, 10)
print(table)

[[ 1  2  3  4  5  6  7  8  9]
 [ 2  4  6  8 10 12 14 16 18]
 [ 3  6  9 12 15 18 21 24 27]
 [ 4  8 12 16 20 24 28 32 36]
 [ 5 10 15 20 25 30 35 40 45]
 [ 6 12 18 24 30 36 42 48 54]
 [ 7 14 21 28 35 42 49 56 63]
 [ 8 16 24 32 40 48 56 64 72]
 [ 9 18 27 36 45 54 63 72 81]]


In [20]:
s = "Hello World!" + word2

word2 = "banana"

NameError: name 'word2' is not defined